# 🏥 MedRAX Reproduction: Local Multimodal Vision (`qwen3-vl:4b-instruct`) on ChestAgentBench
### Reproducing "MedRAX: Medical Reasoning Agent for Chest X-ray" (ICML 2025)
This notebook performs **100% Local, Unlimited Multimodal Reproduction** of the MedRAX benchmark using **`qwen3-vl:4b-instruct`** via Ollama on Google Colab GPU.

| Component | Specification |
| :--- | :--- |
| **Benchmark** | **ChestAgentBench** (2,500 expert-curated clinical questions across 7 clinical axes from Eurorad) |
| **Vision & Agent Model** | **`qwen3-vl:4b-instruct`** via Ollama (Fast Non-Thinking Instruct VLM + Native Tool Calling) |
| **Cost & Rate Limits** | **100% Free, Zero API Keys, Zero Rate Limits, Never Blocked** (Runs 100% on Colab GPU) |
| **VRAM Optimization** | `~3.8 GB` (`qwen3-vl:4b-instruct` Q4_K_M) + `~8.5 GB` (4 MedRAX Tools in 16-bit) = **`~12.3 / 15.0 GB`** (Fits T4 15GB & L4 22.5GB) |
| **Agent Tools** | **CheXagent-2-3B VQA**, TorchXRayVision Classifier & Segmentation, Report Generator |
| **Storage** | Google Drive persistent cache (batch commits every 10 questions with `os.fsync`) |


## 1. ⚙️ Hardware & GPU Environment Diagnostics
Verify the Colab runtime environment (CUDA availability, GPU model, and VRAM).


In [1]:
# Check GPU availability and specifications
import os
import sys
import torch

print(f"Python Version : {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Detected GPU   : {device_name} ({vram_gb:.2f} GB VRAM)")
    print("🚀 Local GPU ready for qwen3-vl:4b-instruct + MedRAX clinical vision tools!")
else:
    print("⚠️ WARNING: No GPU detected! Please enable GPU in Runtime -> Change runtime type.")

!nvidia-smi


Python Version : 3.13.16
PyTorch Version: 2.14.1+cu130
CUDA Available : True
Detected GPU   : NVIDIA L4 (22.03 GB VRAM)
🚀 Local GPU ready for qwen3-vl:4b-instruct + MedRAX clinical vision tools!
Thu Nov 19 03:04:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   43C    P8             13W /  

## 2. 💾 Google Drive Mounting & Persistent Storage
Mount Google Drive to persist downloaded datasets, model checkpoints, and evaluation results.
Batch commits ensure that results are flushed directly to Google Drive every 10 questions.


In [2]:
# Mount Google Drive
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/MedRAX_Thesis')

PERSISTENT_DIRS = {
    "checkpoints": DRIVE_ROOT / "checkpoints",
    "data": DRIVE_ROOT / "data" / "chestagentbench",
    "results": DRIVE_ROOT / "results",
    "logs": DRIVE_ROOT / "logs",
}

for name, path in PERSISTENT_DIRS.items():
    path.mkdir(parents=True, exist_ok=True)
    print(f"📁 Verified persistent directory: {path}")

# Set environment variables
os.environ["MEDRAX_MODEL_DIR"] = str(PERSISTENT_DIRS["checkpoints"])
os.environ["BENCHMARK_DATA_DIR"] = str(PERSISTENT_DIRS["data"])
os.environ["MEDRAX_RESULTS_DIR"] = str(PERSISTENT_DIRS["results"])
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

print("\n✅ Environment variables configured successfully.")


Mounted at /content/drive
📁 Verified persistent directory: /content/drive/MyDrive/MedRAX_Thesis/checkpoints
📁 Verified persistent directory: /content/drive/MyDrive/MedRAX_Thesis/data/chestagentbench
📁 Verified persistent directory: /content/drive/MyDrive/MedRAX_Thesis/results
📁 Verified persistent directory: /content/drive/MyDrive/MedRAX_Thesis/logs

✅ Environment variables configured successfully.


## 3. 🐙 Clone Reproduction Repository from GitHub
Clone the reproduction suite from GitHub. If already present, pull the latest code updates.


In [3]:
# Configure GitHub repository
import os
from pathlib import Path

REPO_URL = "https://github.com/Uyhuynh23/MedRAX-ChestAgentBench.git"
LOCAL_REPO_DIR = Path("/content/MedRAX-ChestAgentBench")

if not LOCAL_REPO_DIR.exists():
    print(f"Cloning {REPO_URL} into {LOCAL_REPO_DIR} ...")
    !git clone {REPO_URL} {LOCAL_REPO_DIR}
else:
    print(f"Repository exists. Pulling latest updates...")
    %cd {LOCAL_REPO_DIR}
    !git pull origin main

%cd {LOCAL_REPO_DIR}
!python scripts/setup_drive.py


Cloning https://github.com/Uyhuynh23/MedRAX-ChestAgentBench.git into /content/MedRAX-ChestAgentBench ...
Cloning into '/content/MedRAX-ChestAgentBench'...
remote: Enumerating objects: 344, done.
remote: Counting objects: 100% (344/344), done.
remote: Compressing objects: 100% (266/266), done.
remote: Total 344 (delta 170), reused 238 (delta 72), pack-reused 0 (from 0)
Receiving objects: 100% (344/344), 1.24 MiB | 18.35 MiB/s, done.
Resolving deltas: 100% (170/170), done.
/content/MedRAX-ChestAgentBench
[Drive Setup] Running in Google Colab environment.
[Drive Setup] Google Drive is already mounted.

 Initializing Persistent Google Drive Architecture
 Root: /content/drive/MyDrive/MedRAX_Thesis
  [OK] Directory ready: /content/drive/MyDrive/MedRAX_Thesis/checkpoints
  [OK] Directory ready: /content/drive/MyDrive/MedRAX_Thesis/cache/huggingface
  [OK] Directory ready: /content/drive/MyDrive/MedRAX_Thesis/cache/torch
  [OK] Directory ready: /content/drive/MyDrive/MedRAX_Thesis/data/chestag

## 4. 📦 Install Dependencies & Specialized Radiology Packages
Install PyTorch, LangChain, TorchXRayVision, and Transformers.


In [4]:
%cd /content/MedRAX-ChestAgentBench

# Install project dependencies
!pip install -q -r requirements.txt
!pip install -q torchxrayvision

print("\n✅ All dependencies installed successfully.")


/content/MedRAX-ChestAgentBench
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.2 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
grpc-google-iam-v1 0.14.6 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-datastore 2.27.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-bigquery-storage 2.43.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-bigquery-connection 1.23.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 wh

## 5. 👁️ Install Ollama & Pull Local Multimodal Model (`qwen3-vl:4b-instruct`)

Why **`qwen3-vl:4b-instruct`** via Ollama is ideal for MedRAX on Colab:
1. **Fast Instruct Variant (No Hidden Thinking Delay)**: Responds in ~2–4 seconds directly into `message.content` (unlike the default `qwen3-vl:4b` thinking tag which spends ~2 minutes generating hidden `<think>` tokens).
2. **Native Vision Encoder**: Directly reads chest X-ray images (`.jpg`, `.png`) unlike text-only `qwen2.5:3b`.
3. **Native Tool Calling (`{{ .Tools }}`)**: Supports LangChain `.bind_tools()` out of the box.
4. **VRAM Optimized**: Uses `OLLAMA_FLASH_ATTENTION=1` and `OLLAMA_KV_CACHE_TYPE=q8_0` to fit comfortably alongside MedRAX's 4 tools on a 15.0 GB T4 GPU.


In [5]:
# -------------------------------------------------------------
# Install Ollama + Configure Low-VRAM Flash Attention + Pull qwen3-vl:4b-instruct
# -------------------------------------------------------------
import os
import time

# 1. Install zstd & pciutils so Ollama extracts and detects NVIDIA GPU properly
!apt-get update -qq && apt-get install -y -qq zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Configure memory-saving flags for Ollama (keeps VRAM usage ~3.8 GB on T4)
os.environ["OLLAMA_FLASH_ATTENTION"] = "1"
os.environ["OLLAMA_KV_CACHE_TYPE"] = "q8_0"
os.environ["OLLAMA_NUM_PARALLEL"] = "1"
os.environ["OLLAMA_MAX_LOADED_MODELS"] = "1"

# 3. Start Ollama server in background with memory optimization flags
!pkill -f "ollama serve" || true
!OLLAMA_FLASH_ATTENTION=1 OLLAMA_KV_CACHE_TYPE=q8_0 OLLAMA_NUM_PARALLEL=1 OLLAMA_MAX_LOADED_MODELS=1 nohup ollama serve > /content/ollama.log 2>&1 &
time.sleep(5)

# 4. Pull qwen3-vl:4b-instruct (Fast non-thinking Instruct variant, ~3.3 GB)
!ollama pull qwen3-vl:4b-instruct
!ollama list

print("\n✅ qwen3-vl:4b-instruct is installed and serving locally at http://localhost:11434/v1")


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package pci.ids.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../pci.ids_0.0~2024.03.31-1ubuntu0.1_all.deb ...
Unpacking pci.ids (0.0~2024.03.31-1ubuntu0.1) ...
Selecting previously unselected package libpci3:amd64.
Preparing to unpack .../libpci3_1%3a3.10.0-2build1_amd64.deb ...
Unpacking libpci3:amd64 (1:3.10.0-2build1) ...
Selecting previously unselected package pciutils.
Preparing to unpack .../pciutils_1%3a3.10.0-2build1_amd64.deb ...
Unpacking pciutils (1:3.10.0-2build1) ...
Selecting previously unselected package zstd.
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up pci.ids (0.0~2024.03.31-1ubuntu0.1) ...
Setting up libpci3:amd64 (1:3.10.0-2bui

In [7]:
# -------------------------------------------------------------
# 🧪 Strict Visual Ground-Truth Proof & Tool-Calling Diagnostic
# -------------------------------------------------------------
# Draws a secret code ("7492") ONLY inside the image pixels (never in the prompt)
# and verifies that qwen3-vl:4b-instruct actually reads it with its Vision Encoder!
import time
import base64
import io
from PIL import Image, ImageDraw
from openai import OpenAI

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
MODEL_NAME = "qwen3-vl:4b-instruct"

# Draw secret visual code "7492" onto an image canvas
test_img = Image.new("RGB", (320, 120), color=(20, 20, 20))
draw = ImageDraw.Draw(test_img)
draw.rectangle([10, 10, 310, 110], outline=(220, 220, 220), width=3)
draw.text((40, 45), "SECRET CODE: 7492", fill=(255, 255, 255))

buf = io.BytesIO()
test_img.save(buf, format="JPEG", quality=95)
b64_img = base64.b64encode(buf.getvalue()).decode("utf-8")

print("=======================================================")
print(f" Running Strict Vision & Tool Proof ({MODEL_NAME})")
print("=======================================================\n")

# Test 1: Strict Visual Ground-Truth Proof (Model MUST read '7492' from pixels)
t0 = time.perf_counter()
resp_vis = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{
        "role": "user",
        "content": [
            {"type": "text", "text": "Read the exact 4-digit secret code written inside this image."},
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64_img}"}}
        ]
    }],
    temperature=0.0,
    max_tokens=128,
)
t_vis = time.perf_counter() - t0
vis_txt = " ".join((resp_vis.choices[0].message.content or "").split())

if "7492" in vis_txt:
    print(f"  [Test 1: Multimodal Vision] ✅ PROVEN ({t_vis:.2f}s) -> Model read '7492' from image! Output: {vis_txt}")
else:
    print(f"  [Test 1: Multimodal Vision] ❌ FAILED ({t_vis:.2f}s) -> Expected '7492', got: '{vis_txt}'")

# Test 2: Native Tool Calling Support
t1 = time.perf_counter()
resp_tool = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": "Call chest_xray_classifier on /content/figure_1.jpg"}],
    tools=[{
        "type": "function",
        "function": {
            "name": "chest_xray_classifier",
            "description": "Classifies chest X-ray image for 18 pathologies",
            "parameters": {
                "type": "object",
                "properties": {"image_path": {"type": "string"}},
                "required": ["image_path"]
            }
        }
    }],
    temperature=0.0,
    max_tokens=256,
)
t_tool = time.perf_counter() - t1
t_calls = resp_tool.choices[0].message.tool_calls
if t_calls and t_calls[0].function.name == "chest_xray_classifier":
    print(f"  [Test 2: Native Tool Call ] ✅ PROVEN ({t_tool:.2f}s) -> Tool called: {t_calls[0].function.name}({t_calls[0].function.arguments})")
else:
    print(f"  [Test 2: Native Tool Call ] ❌ FAILED ({t_tool:.2f}s) -> No tool call detected.")


 Running Strict Vision & Tool Proof (qwen3-vl:4b-instruct)

  [Test 1: Multimodal Vision] ✅ PROVEN (119.29s) -> Model read '7492' from image! Output: 7492
  [Test 2: Native Tool Call ] ✅ PROVEN (0.45s) -> Tool called: chest_xray_classifier({"image_path":"/content/figure_1.jpg"})


## 6. 📥 Download ChestAgentBench Dataset & Extract Figures
Download the **ChestAgentBench** benchmark dataset (`wanglab/chest-agent-bench`) from Hugging Face into Google Drive.
If already present on Google Drive, it will verify integrity and skip re-downloading.


In [6]:
# Download / Verify ChestAgentBench in persistent Google Drive
%cd /content/MedRAX-ChestAgentBench
!python scripts/download_benchmark.py --data-dir "$BENCHMARK_DATA_DIR"

!ls -lh "$BENCHMARK_DATA_DIR"
!head -n 2 "$BENCHMARK_DATA_DIR/metadata.jsonl" | cut -c 1-120


/content/MedRAX-ChestAgentBench
 ChestAgentBench Dataset Setup
 Target Directory: /content/drive/MyDrive/MedRAX_Thesis/data/chestagentbench
[Check] Dataset files and figures directory already exist.

----------------- Verification Summary -----------------
 Total Multiple-Choice Questions: 2500 (Expected: 2,500)
 Unique Clinical Cases:           609 (Expected: ~609)
 Image References Checked:        4629/4629
 Case Folders in figures/:        957
--------------------------------------------------------
[SUCCESS] Dataset ready for MedRAX evaluation!

total 790M
drwx------ 959 root root 4.0K Nov 13 16:46 figures
-rw-------   1 root root 785M Nov 13 16:46 figures.zip
-rw-------   1 root root 4.7M Nov 13 16:46 metadata.jsonl
-rw-------   1 root root 3.4K Nov 13 16:46 README.md
{"images": ["figures/11583/figure_1.jpg"], "image_source_urls": ["https://www.eurorad.org/sites/default/files/styles/fig
{"images": ["figures/11583/figure_1.jpg"], "image_source_urls": ["https://www.eurorad.org/sites

## 7. 🔍 Check Saved Progress on Google Drive
Run the cell below anytime to check how many questions are already saved in Google Drive before starting or resuming a run.


In [9]:
# -------------------------------------------------------------
# 📊 Check Saved Progress on Google Drive (100 Cases ~ 400 Questions)
# -------------------------------------------------------------
import json
from pathlib import Path

results_dir = Path(os.environ["MEDRAX_RESULTS_DIR"])
for run_name in ["qwen3vl_baseline_100cases", "qwen3vl_medrax_100cases"]:
    pred_file = results_dir / run_name / "predictions.jsonl"
    if pred_file.exists():
        lines = [json.loads(line) for line in open(pred_file, encoding="utf-8") if line.strip()]
        corr = sum(1 for r in lines if r.get("is_correct"))
        acc = (corr / len(lines) * 100) if lines else 0.0
        print(f"📁 {run_name:<26}: {len(lines):>4} questions saved | Running Acc: {acc:.2f}% ({corr}/{len(lines)})")
    else:
        print(f"📁 {run_name:<26}:    0 questions saved (Not started yet)")


📁 qwen3vl_baseline_100cases :    0 questions saved (Not started yet)
📁 qwen3vl_medrax_100cases   :    0 questions saved (Not started yet)


## 8. 🧪 Benchmark Execution (`qwen3-vl:4b-instruct` | `--max-cases 100` Cochran 95% Subset)

### Statistical & Resource Design (`--max-cases 100` ≈ 400 Questions):
* **Cochran's Formula ($N = 2,500$, 95% Confidence, $\pm 5\%$ Margin of Error)**: Requires $n \ge 333$ questions. Evaluating **100 cases (~400 questions)** exceeds Cochran's threshold and covers all 7 clinical categories (*Detection, Classification, Localization, Comparison, Relationship, Diagnosis, Characterization*).
* **Fast & Disconnect-Proof**: Finishes in ~40–90 minutes on Colab GPU (instead of 8 hours), with batch commits every 10 questions (`os.fsync`) and automatic resume!


In [10]:
# -------------------------------------------------------------
# 🧪 Step 8.1a: Smoke Test - Baseline Vision Mode on 1 Case
# -------------------------------------------------------------
%cd /content/MedRAX-ChestAgentBench

!python scripts/run_evaluation.py \
    --mode baseline \
    --model qwen3-vl:4b-instruct \
    --base-url "http://localhost:11434/v1" \
    --api-key "ollama" \
    --data-dir "$BENCHMARK_DATA_DIR" \
    --output-dir "$MEDRAX_RESULTS_DIR/smoke_test_qwen3vl_baseline" \
    --max-cases 1 \
    --reset


/content/MedRAX-ChestAgentBench
[Reset] --reset active: Clearing previous predictions in /content/drive/MyDrive/MedRAX_Thesis/results/smoke_test_qwen3vl_baseline to evaluate 100% clean under qwen3-vl:4b-instruct.
[Filter] Selected 5 questions across 1 cases (limit: 1).

 Starting ChestAgentBench Evaluation
 Mode: baseline | Model: qwen3-vl:4b-instruct | Total Questions: 5
 Results will be saved to: /content/drive/MyDrive/MedRAX_Thesis/results/smoke_test_qwen3vl_baseline/predictions.jsonl

[1/5] QID: 11583_8353800736258 | Pred: A Gold: B ✗ | Running Acc: 0.00% (0/1)
[2/5] QID: 11583_8353800841102 | Pred: C Gold: C ✓ | Running Acc: 50.00% (1/2)
[3/5] QID: 11583_8353800778861 | Pred: C Gold: C ✓ | Running Acc: 66.67% (2/3)
[4/5] QID: 11583_8353800764059 | Pred: C Gold: B ✗ | Running Acc: 50.00% (2/4)
[5/5] QID: 11583_8353800840529 | Pred: C Gold: C ✓ | Running Acc: 60.00% (3/5)
  [Drive Sync] Final flush: 5 remaining records saved to Google Drive.

 Evaluation Completed!
 Overall Accuracy

In [11]:
# -------------------------------------------------------------
# 🧪 Step 8.1b: Smoke Test - MedRAX Agent Mode on 1 Case
# -------------------------------------------------------------
%cd /content/MedRAX-ChestAgentBench

!python scripts/run_evaluation.py \
    --mode medrax \
    --model qwen3-vl:4b-instruct \
    --base-url "http://localhost:11434/v1" \
    --api-key "ollama" \
    --data-dir "$BENCHMARK_DATA_DIR" \
    --output-dir "$MEDRAX_RESULTS_DIR/smoke_test_qwen3vl_medrax" \
    --max-cases 1 \
    --reset


/content/MedRAX-ChestAgentBench
[Reset] --reset active: Clearing previous predictions in /content/drive/MyDrive/MedRAX_Thesis/results/smoke_test_qwen3vl_medrax to evaluate 100% clean under qwen3-vl:4b-instruct.
[Filter] Selected 5 questions across 1 cases (limit: 1).
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
[Init] Initializing MedRAX Agent with tools: ['ChestXRayClassifierTool', 'ChestXRaySegmentationTool', 'ChestXRayReportGeneratorTool', 'XRayVQATool'] on cuda...
If this fails you can run `wget https://github.com/mlmed/torchxrayvision/releases/download/v1/nih-pc-chex-mimic_ch-google-openi-kaggle-densenet121-d121-tw-lr001-rot45-tr15-sc15-seed0-best.pt -O /root/.torchxrayvision/models_data/nih-pc-chex-mimic_ch-google-openi-kaggle-de

In [8]:
# -------------------------------------------------------------
# 📊 Option A: Multimodal Baseline Evaluation (100 Cases ~ 400 Questions)
# -------------------------------------------------------------
# qwen3-vl:4b-instruct reads chest radiographs directly with its native vision encoder.
# Batch commits every 10 questions with os.fsync directly to Google Drive.
# Safe to re-run anytime (NO --reset flag, so it auto-resumes!).
%cd /content/MedRAX-ChestAgentBench

!python scripts/run_evaluation.py \
    --mode baseline \
    --model qwen3-vl:4b-instruct \
    --base-url "http://localhost:11434/v1" \
    --api-key "ollama" \
    --data-dir "$BENCHMARK_DATA_DIR" \
    --output-dir "$MEDRAX_RESULTS_DIR/qwen3vl_baseline_100cases" \
    --max-cases 100


/content/MedRAX-ChestAgentBench
[Resume] Found 2 questions already evaluated. Skipping them.
[Filter] Selected 420 questions across 100 cases (limit: 100).

 Starting ChestAgentBench Evaluation
 Mode: baseline | Model: qwen3-vl:4b-instruct | Total Questions: 420
 Results will be saved to: /content/drive/MyDrive/MedRAX_Thesis/results/qwen3vl_baseline_100cases/predictions.jsonl

[3/420] QID: 11583_8353800778861 | Pred: C Gold: C ✓ | Running Acc: 100.00% (3/3)
[4/420] QID: 11583_8353800764059 | Pred: C Gold: B ✗ | Running Acc: 75.00% (3/4)
[5/420] QID: 11583_8353800840529 | Pred: A Gold: C ✗ | Running Acc: 60.00% (3/5)
[6/420] QID: 13388_8353800587274 | Pred: A Gold: C ✗ | Running Acc: 50.00% (3/6)
[7/420] QID: 13388_8353800651470 | Pred: E Gold: C ✗ | Running Acc: 42.86% (3/7)
[8/420] QID: 13388_8353800704178 | Pred: E Gold: E ✓ | Running Acc: 50.00% (4/8)
[9/420] QID: 10098_8353800802244 | Pred: A Gold: C ✗ | Running Acc: 44.44% (4/9)
[10/420] QID: 10098_8353800828916 | Pred: A Gold: B 

In [9]:
# -------------------------------------------------------------
# 🏥 Option B: MedRAX Agent Evaluation (100 Cases ~ 400 Questions)
# -------------------------------------------------------------
# qwen3-vl:4b-instruct sees the radiographs + coordinates local clinical tools (CheXagent, TorchXRayVision).
# Batch commits every 10 questions with os.fsync directly to Google Drive.
# Safe to re-run anytime (NO --reset flag, so it auto-resumes!).
%cd /content/MedRAX-ChestAgentBench

!python scripts/run_evaluation.py \
    --mode medrax \
    --model qwen3-vl:4b-instruct \
    --base-url "http://localhost:11434/v1" \
    --api-key "ollama" \
    --data-dir "$BENCHMARK_DATA_DIR" \
    --output-dir "$MEDRAX_RESULTS_DIR/qwen3vl_medrax_100cases" \
    --max-cases 100


/content/MedRAX-ChestAgentBench
[Resume] Found 417 questions already evaluated. Skipping them.
[Filter] Selected 420 questions across 100 cases (limit: 100).
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
[Init] Initializing MedRAX Agent with tools: ['ChestXRayClassifierTool', 'ChestXRaySegmentationTool', 'ChestXRayReportGeneratorTool', 'XRayVQATool'] on cuda...
Loading checkpoint shards: 100% 3/3 [00:52<00:00, 17.41s/it]
Agent using custom OpenAI-compatible endpoint: http://localhost:11434/v1 (model: qwen3-vl:4b-instruct)
Agent initialized

 Starting ChestAgentBench Evaluation
 Mode: medrax | Model: qwen3-vl:4b-instruct | Total Questions: 420
 Results will be saved to: /content/drive/MyDrive/MedRAX_Thesis/results/qwen3vl_medrax_100cases

## 9. 📊 Comparative Analytics: Baseline vs MedRAX Agent (100 Cases)
Generate side-by-side comparison tables and accuracy breakdown charts across the 7 clinical axes:
* **Detection**, **Classification**, **Localization**, **Comparison**, **Relationship**, **Diagnosis**, **Characterization**


In [14]:
# -------------------------------------------------------------
# ⚖️ Compare Baseline vs MedRAX Agent Performance (100 Cases)
# -------------------------------------------------------------
%cd /content/MedRAX-ChestAgentBench

!python experiments/compare_runs.py \
    "$MEDRAX_RESULTS_DIR/qwen3vl_baseline_100cases/predictions.jsonl" \
    "$MEDRAX_RESULTS_DIR/qwen3vl_medrax_100cases/predictions.jsonl" \
    --output-dir "$MEDRAX_RESULTS_DIR/comparisons_qwen3vl_100cases"

# Display generated comparison plot
from IPython.display import Image, display
chart_path = Path(os.environ["MEDRAX_RESULTS_DIR"]) / "comparisons_qwen3vl_100cases" / "accuracy_breakdown.png"
if chart_path.exists():
    display(Image(str(chart_path)))


/content/MedRAX-ChestAgentBench


KeyboardInterrupt: 

## 10. 🔄 Synchronize Results & Updates to GitHub
Push any notebook changes or evaluation summaries back to your GitHub repository.


In [ ]:
# Configure Git credentials in Colab
!git config --global user.name "Uy Huynh"
!git config --global user.email "huynhtanuy123@gmail.com"

# Check Git status
%cd /content/MedRAX-ChestAgentBench
!git status -s


In [17]:
# -------------------------------------------------------------
# 🔄 Pull Code Fix & Restart Ollama with 8192 Context Length
# -------------------------------------------------------------
%cd /content/MedRAX-ChestAgentBench
!git pull origin main

import os, time, subprocess, urllib.request, json

# 1. Tắt tiến trình Ollama cũ (4096 context) để bật lại với 8192 context
!pkill -f "ollama serve" || true
time.sleep(2)

ollama_env = os.environ.copy()
ollama_env["OLLAMA_KEEP_ALIVE"] = "-1"
ollama_env["OLLAMA_CONTEXT_LENGTH"] = "8192"
ollama_env["OLLAMA_FLASH_ATTENTION"] = "1"
ollama_env["OLLAMA_KV_CACHE_TYPE"] = "q8_0"
ollama_env["OLLAMA_NUM_PARALLEL"] = "1"
ollama_env["OLLAMA_MAX_LOADED_MODELS"] = "1"

log_file = open("/content/ollama.log", "a")
subprocess.Popen(
    ["ollama", "serve"],
    env=ollama_env,
    stdout=log_file,
    stderr=log_file,
    start_new_session=True,
)
time.sleep(5)

# 2. Ghim qwen3-vl:4b-instruct vào bộ nhớ với num_ctx = 8192 và keep_alive = -1
req = urllib.request.Request(
    "http://localhost:11434/api/generate",
    data=json.dumps({
        "model": "qwen3-vl:4b-instruct",
        "prompt": "hi",
        "keep_alive": -1,
        "options": {"num_ctx": 8192, "num_predict": 1}
    }).encode("utf-8"),
    headers={"Content-Type": "application/json"},
)
with urllib.request.urlopen(req, timeout=120) as resp:
    print("🔒 Ollama restarted with 8192 Context & pinned model:", resp.status == 200)

!ollama ps

/content/MedRAX-ChestAgentBench
remote: Enumerating objects: 19, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 10 (delta 8), reused 10 (delta 8), pack-reused 0 (from 0)
Unpacking objects: 100% (10/10), 2.31 KiB | 473.00 KiB/s, done.
From https://github.com/Uyhuynh23/MedRAX-ChestAgentBench
 * branch            main       -> FETCH_HEAD
   0fcc6c3..c895614  main       -> origin/main
Updating 0fcc6c3..c895614
Fast-forward
 main.py                                            |  2 +-
 medrax/agent/agent.py                              | 16 ++------
 ...AX_ChestAgentBench_Qwen3VL_Instruct_Colab.ipynb | 47 +++++++++++++++++-----
 scripts/run_evaluation.py                          | 41 ++++++++++++++-----
 4 files changed, 74 insertions(+), 32 deletions(-)
^C
🔒 Ollama restarted with 8192 Context & pinned model: True
NAME                    ID              SIZE      PROCESSOR    CONTEXT    RUNNER      UNTIL   
qwen3-vl:4b-instruct